# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [4]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [5]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [8]:
df_trips.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)



## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

Au debut voir ce qu'on a 

In [9]:
df_trips.printSchema()
df_trips.show(5)

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+---

In [12]:
from pyspark.sql import functions as F

df_trips = df_trips.withColumn(
    "trip_id",
    F.monotonically_increasing_id()
)

df_trips.select("trip_id", "passenger_count").show(5)

+-----------+---------------+
|    trip_id|passenger_count|
+-----------+---------------+
|60129542144|            1.0|
|60129542145|            1.0|
|60129542146|            3.0|
|60129542147|            5.0|
|60129542148|            5.0|
+-----------+---------------+
only showing top 5 rows


In [13]:
df_trips.select(
    "trip_id",
    "passenger_count",
    "trip_distance"
).orderBy(
    F.col("passenger_count").desc()
).show(5)

+-----------+---------------+-------------+
|    trip_id|passenger_count|trip_distance|
+-----------+---------------+-------------+
|60131554242|            9.0|          0.0|
|60136828827|            9.0|          0.0|
|60132426139|            9.0|          0.0|
|60130838431|            9.0|          0.0|
|60134076851|            9.0|          0.0|
+-----------+---------------+-------------+
only showing top 5 rows


ON a ajouté un ientifiant a chaque trajet et ensuite trié par le nb de passager de manière décroissante.

In [17]:
df_trips.agg(
    F.avg("passenger_count").alias("average_passenger_count")
).show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



Maintenant on cherche les trajrtr du plus court au plus long en distance puis en durée.

In [18]:

df_trips.select(
    "trip_id", "trip_distance"
).orderBy(
    F.col("trip_distance").asc_nulls_last()
).show(5)


df_trips.select(
    "trip_id", "trip_distance"
).orderBy(
    F.col("trip_distance").desc_nulls_last()
).show(5)

+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|60129542149|          0.0|
|60129542150|          0.0|
|60129542172|          0.0|
|60129542148|          0.0|
|60129542146|          0.0|
+-----------+-------------+
only showing top 5 rows
+-----------+-------------+
|    trip_id|trip_distance|
+-----------+-------------+
|60135616235|        831.8|
|60133828777|        700.7|
|60136313129|       214.01|
|60134249678|       211.36|
|60134423929|       201.27|
+-----------+-------------+
only showing top 5 rows


In [20]:
df_trips.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- trip_id: long (nullable = false)



In [21]:
df_trips = df_trips.withColumn(
    "trip_duration_minutes",
    (
        F.col("tpep_dropoff_datetime").cast("timestamp").cast("long")
        - F.col("tpep_pickup_datetime").cast("timestamp").cast("long")
    ) / 60
)

df_trips.select("trip_id", "trip_duration_minutes").show(5)

+-----------+---------------------+
|    trip_id|trip_duration_minutes|
+-----------+---------------------+
|60129542144|    6.666666666666667|
|60129542145|                 19.2|
|60129542146|    4.166666666666667|
|60129542147|   3.3333333333333335|
|60129542148|                  1.6|
+-----------+---------------------+
only showing top 5 rows


on cherche le jour avec le plus de trajrts et celui avec le moins

In [22]:
trips_per_day = (
    df_trips
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
    )
    .groupBy("trip_date")
    .count()
)

Ca cest pour regrouper le nb de trajrt par jour 

In [23]:
trips_per_day.orderBy(
    F.col("count").desc(),
    F.col("trip_date").asc()
).show(5)

+----------+------+
| trip_date| count|
+----------+------+
|2019-01-25|292499|
|2019-01-11|291714|
|2019-01-31|284625|
|2019-01-17|284580|
|2019-01-24|281959|
+----------+------+
only showing top 5 rows


In [24]:
trips_per_day.orderBy(
    F.col("count").asc(),
    F.col("trip_date").asc()
).show(5) 
#les jurs kes moins fréquentes 

+----------+------+
| trip_date| count|
+----------+------+
|2019-01-01|189432|
|2019-01-21|192826|
|2019-01-02|198737|
|2019-01-20|203114|
|2019-01-06|208823|
+----------+------+
only showing top 5 rows


In [25]:
#les heures avec le plus et le moins de départs en regroupant tous les jours de janvier 2019
trips_per_hour = (
    df_trips
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .count()
)

In [26]:
trips_per_hour.orderBy(
    F.col("count").desc(),
    F.col("pickup_hour").asc()
).show(5)
#les heures les plus fréquentés

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515374|
|         19|475152|
|         17|468407|
|         15|452679|
|         14|433115|
+-----------+------+
only showing top 5 rows


In [27]:
trips_per_hour.orderBy(
    F.col("count").asc(),
    F.col("pickup_hour").asc()
).show(5)# les heures le smoins fréquentés 

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          4| 61423|
|          5| 75532|
|          3| 78084|
|          2|109413|
|          1|149242|
+-----------+------+
only showing top 5 rows


In [28]:

#mooyenne des trajets par semaine 
trips_per_weekday = (
    trips_per_day
    .withColumn("weekday", F.dayofweek("trip_date"))
    .groupBy("weekday")
    .agg(
        F.avg("count").alias("average_daily_trips"),
        F.count("*").alias("number_of_days")
    )
)

trips_per_weekday.orderBy(
    F.col("average_daily_trips").desc()
).show()

+-------+-------------------+--------------+
|weekday|average_daily_trips|number_of_days|
+-------+-------------------+--------------+
|      6|           271787.5|             4|
|      5|           271398.4|             5|
|      4|           253045.8|             5|
|      7|          252494.75|             4|
|      3|           241815.2|             5|
|      2|           226941.0|             4|
|      1|           214972.5|             4|
+-------+-------------------+--------------+



In [29]:
tips_per_passenger_count = (
    df_trips
    .filter(
        (F.col("passenger_count") > 0)
        & (F.col("tip_amount") >= 0)
        & (F.col("payment_type") == 1)
    )
    .groupBy("passenger_count")
    .agg(
        F.avg("tip_amount").alias("average_tip"),
        F.count("*").alias("number_of_trips")
    )
)

tips_per_passenger_count.orderBy("passenger_count").show()

+---------------+------------------+---------------+
|passenger_count|       average_tip|number_of_trips|
+---------------+------------------+---------------+
|            1.0|2.5339521622183447|        3936888|
|            2.0| 2.614486572689903|         781318|
|            3.0| 2.585738670425347|         218521|
|            4.0|2.6029785593256283|          92068|
|            5.0|2.6182445012300475|         231279|
|            6.0| 2.609179752008406|         142908|
|            7.0| 11.30090909090909|             11|
|            8.0|  6.96074074074074|             27|
|            9.0|           3.50625|              8|
+---------------+------------------+---------------+



In [ ]:
#trajet plus long pourboire plus elevé ? 
tips_per_distance = (
    df_trips
    .filter(
        (F.col("trip_distance") > 0)
        & (F.col("tip_amount") >= 0)
        & (F.col("payment_type") == 1)
    )
    .withColumn(
        "distance_start",
        F.floor(F.col("trip_distance") / 5) * 5
    )
    .groupBy("distance_start")
    .agg(
        F.avg("tip_amount").alias("average_tip"),
        F.count("*").alias("number_of_trips")
    )
)

tips_per_distance.orderBy("distance_start").show(20)

+--------------+------------------+---------------+
|distance_start|       average_tip|number_of_trips|
+--------------+------------------+---------------+
|             0|1.9303702471653787|        4701481|
|             5| 4.683769193721003|         433931|
|            10| 6.722388618809453|         155098|
|            15| 9.715289603573051|         135133|
|            20| 9.888293642716777|          29478|
|            25|11.045881203595075|           5118|
|            30| 15.07368755676656|           1101|
|            35| 18.72457528957527|            518|
|            40|24.921449999999997|            200|
|            45|22.811666666666667|            144|
|            50|27.141624999999994|             80|
|            55| 29.04509433962264|             53|
|            60|24.264374999999998|             32|
|            65|15.436842105263159|             19|
|            70|19.749999999999996|             12|
|            75|          33.36875|              8|
|           

In [31]:
#le supplement extra le plus élevé et le rajet associé 
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "extra"
).orderBy(
    F.col("extra").desc_nulls_last()
).show(5, truncate=False)

+-----------+--------------------+-------------+-----------+------+
|trip_id    |tpep_pickup_datetime|trip_distance|fare_amount|extra |
+-----------+--------------------+-------------+-----------+------+
|60134865627|2019-01-23 08:58:09 |0.0          |355676.98  |535.38|
|60136995374|2019-01-31 10:06:09 |0.0          |4.5        |23.04 |
|60129853196|2019-01-02 16:33:28 |17.23        |52.0       |18.5  |
|60131997230|2019-01-11 16:08:48 |15.78        |49.0       |18.5  |
|60129676693|2019-01-01 16:09:32 |13.4         |39.5       |18.5  |
+-----------+--------------------+-------------+-----------+------+
only showing top 5 rows


In [ ]:
suspect_trips = df_trips.filter(
    (F.col("passenger_count") <= 0)
    | (F.col("trip_distance") <= 0)
    | (F.col("trip_duration_minutes") <= 0)
    | (F.col("fare_amount") < 0)
)

suspect_trips.select(
    "trip_id",
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount"
).show(20, truncate=False)


"it seems not normal because of the trip distance and the duration in minutes. If a trop is 0m why is the duration different from 0"

+-----------+---------------+-------------+---------------------+-----------+
|trip_id    |passenger_count|trip_distance|trip_duration_minutes|fare_amount|
+-----------+---------------+-------------+---------------------+-----------+
|60129542146|3.0            |0.0          |4.166666666666667    |4.5        |
|60129542147|5.0            |0.0          |3.3333333333333335   |3.5        |
|60129542148|5.0            |0.0          |1.6                  |52.0       |
|60129542149|5.0            |0.0          |2.6166666666666667   |3.5        |
|60129542150|5.0            |0.0          |4.1                  |52.0       |
|60129542172|3.0            |0.0          |0.0                  |6.5        |
|60129542300|0.0            |5.3          |0.95                 |2.5        |
|60129542372|0.0            |18.0         |30.966666666666665   |52.0       |
|60129542373|0.0            |8.9          |31.316666666666666   |29.5       |
|60129542442|0.0            |1.0          |10.566666666666666   

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [33]:
zone_url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zone_lookup.csv"
)

response = requests.get(zone_url, timeout=60)
response.raise_for_status()

with open("taxi_zone_lookup.csv", "wb") as file:
    file.write(response.content)

In [34]:
df_zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("taxi_zone_lookup.csv")
)

df_zones.show(5, truncate=False)
df_zones.printSchema()

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [35]:
df_trips.PULocationID = df_zones.LocationID

In [37]:
pickup_zones = df_zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough")
)

dropoff_zones = df_zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough")
)

In [38]:
df_trips_zones = (
    df_trips
    .join(pickup_zones, on="PULocationID", how="left")
    .join(dropoff_zones, on="DOLocationID", how="left")
)

df_trips_zones.select(
    "trip_id", "pickup_borough", "dropoff_borough"
).show(5)

+-----------+--------------+---------------+
|    trip_id|pickup_borough|dropoff_borough|
+-----------+--------------+---------------+
|60129542144|     Manhattan|      Manhattan|
|60129542145|     Manhattan|      Manhattan|
|60129542146|     Manhattan|      Manhattan|
|60129542147|        Queens|         Queens|
|60129542148|        Queens|         Queens|
+-----------+--------------+---------------+
only showing top 5 rows


In [39]:
#les heures les plus et moins frequentes pour chaque borough 
#d'aborf je compte les departs"
trips_by_borough_hour = (
    df_trips_zones
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
        & F.col("pickup_borough").isin(
            "Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"
        )
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_borough", "pickup_hour")
    .count()
)

In [40]:
#on cherche le max et le min dans chaque borough

from pyspark.sql.window import Window

borough_window = Window.partitionBy("pickup_borough")

ranked_hours = (
    trips_by_borough_hour
    .withColumn(
        "max_trips",
        F.max("count").over(borough_window)
    )
    .withColumn(
        "min_trips",
        F.min("count").over(borough_window)
    )
)

In [41]:
#on afffiche les heures les plus fréquentés
ranked_hours.filter(
    F.col("count") == F.col("max_trips")
).select(
    "pickup_borough", "pickup_hour", "count"
).orderBy("pickup_borough", "pickup_hour").show()

+--------------+-----------+------+
|pickup_borough|pickup_hour| count|
+--------------+-----------+------+
|         Bronx|          7|  1803|
|      Brooklyn|          8|  6935|
|     Manhattan|         18|471524|
|        Queens|         16| 29880|
| Staten Island|          8|    36|
+--------------+-----------+------+



In [42]:
#les moins frequentés 
ranked_hours.filter(
    F.col("count") == F.col("min_trips")
).select(
    "pickup_borough", "pickup_hour", "count"
).orderBy("pickup_borough", "pickup_hour").show()

+--------------+-----------+-----+
|pickup_borough|pickup_hour|count|
+--------------+-----------+-----+
|         Bronx|          2|  225|
|         Bronx|          3|  225|
|      Brooklyn|          3| 1919|
|     Manhattan|          4|53446|
|        Queens|          3| 3084|
| Staten Island|          1|    3|
+--------------+-----------+-----+



In [43]:

#on compte les departs pour chaque borough
daily_trips_by_borough = (
    df_trips_zones
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
        & F.col("pickup_borough").isin(
            "Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"
        )
    )
    .groupBy("pickup_borough", "trip_date")
    .count()
)

In [44]:
#mmoyenne part jour de la semaine 
borough_weekdays = (
    daily_trips_by_borough
    .withColumn("weekday", F.dayofweek("trip_date"))
    .groupBy("pickup_borough", "weekday")
    .agg(
        F.avg("count").alias("average_daily_trips")
    )
)

In [45]:
#garder le jour le plus fréquenté
borough_window = Window.partitionBy("pickup_borough")

busiest_weekdays = borough_weekdays.withColumn(
    "max_average",
    F.max("average_daily_trips").over(borough_window)
)

busiest_weekdays.filter(
    F.col("average_daily_trips") == F.col("max_average")
).select(
    "pickup_borough", "weekday", "average_daily_trips"
).orderBy("pickup_borough", "weekday").show()

+--------------+-------+-------------------+
|pickup_borough|weekday|average_daily_trips|
+--------------+-------+-------------------+
|         Bronx|      6|              666.5|
|      Brooklyn|      6|             3273.0|
|     Manhattan|      6|           246224.0|
|        Queens|      2|            16662.0|
| Staten Island|      6|               16.0|
+--------------+-------+-------------------+



In [46]:
#on cherche la distance moyenne des trajets par arrondissement de départ
average_distance_by_borough = (
    df_trips_zones
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
        & F.col("pickup_borough").isin(
            "Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"
        )
    )
    .groupBy("pickup_borough")
    .agg(
        F.avg("trip_distance").alias("average_distance"),
        F.count("trip_distance").alias("number_of_trips")
    )
)

average_distance_by_borough.orderBy(
    F.col("average_distance").desc()
).show()

+--------------+------------------+---------------+
|pickup_borough|  average_distance|number_of_trips|
+--------------+------------------+---------------+
| Staten Island|12.503601108033246|            361|
|        Queens|11.283685060696447|         471113|
|         Bronx| 7.234111098803697|          18056|
|      Brooklyn|   4.7876346086881|          91896|
|     Manhattan|2.2286369894240687|        6950511|
+--------------+------------------+---------------+



In [47]:
#tarifmoyen des trajets par arrondissemen de départ
average_fare_by_borough = (
    df_trips_zones
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
        & F.col("pickup_borough").isin(
            "Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"
        )
    )
    .groupBy("pickup_borough")
    .agg(
        F.avg("fare_amount").alias("average_fare"),
        F.count("fare_amount").alias("number_of_trips")
    )
)

average_fare_by_borough.orderBy(
    F.col("average_fare").desc()
).show()

+--------------+------------------+---------------+
|pickup_borough|      average_fare|number_of_trips|
+--------------+------------------+---------------+
| Staten Island|45.289861495844896|            361|
|        Queens|35.144710419789384|         471113|
|         Bronx|26.270240917147586|          18056|
|      Brooklyn| 18.64909843736217|          91896|
|     Manhattan|  10.7923875395659|        6950511|
+--------------+------------------+---------------+



In [48]:
#trajetr avec le tarif le plus élevé et plus faible ainsi que arrondissement de départ
# on garde les trajets de depart de janvier 22019
january_trips_zones = (
    df_trips_zones
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .filter(
        (F.col("trip_date") >= F.lit("2019-01-01").cast("date"))
        & (F.col("trip_date") < F.lit("2019-02-01").cast("date"))
    )
)


In [49]:
#tarif plus eleve puis mon élevé 
january_trips_zones.select(
    "trip_id",
    "pickup_borough",
    "dropoff_borough",
    "trip_distance",
    "fare_amount"
).orderBy(
    F.col("fare_amount").desc_nulls_last()
).show(5, truncate=False)

+-----------+--------------+---------------+-------------+-----------+
|trip_id    |pickup_borough|dropoff_borough|trip_distance|fare_amount|
+-----------+--------------+---------------+-------------+-----------+
|60132041799|Manhattan     |Manhattan      |2.4          |623259.86  |
|60134865627|Manhattan     |Unknown        |0.0          |355676.98  |
|60131702115|Unknown       |Unknown        |0.0          |36090.3    |
|60131434925|Unknown       |Unknown        |0.0          |34674.65   |
|60131191595|Unknown       |Unknown        |0.0          |33023.53   |
+-----------+--------------+---------------+-------------+-----------+
only showing top 5 rows


In [50]:
january_trips_zones.select(
    "trip_id",
    "pickup_borough",
    "dropoff_borough",
    "trip_distance",
    "fare_amount"
).orderBy(
    F.col("fare_amount").asc_nulls_last()
).show(5, truncate=False)

+-----------+--------------+---------------+-------------+-----------+
|trip_id    |pickup_borough|dropoff_borough|trip_distance|fare_amount|
+-----------+--------------+---------------+-------------+-----------+
|60134432793|Queens        |Queens         |0.0          |-362.0     |
|60135850344|N/A           |N/A            |0.0          |-320.0     |
|60129599238|Bronx         |Bronx          |0.0          |-300.0     |
|60136769998|N/A           |N/A            |0.0          |-300.0     |
|60130747032|Queens        |Queens         |0.75         |-284.0     |
+-----------+--------------+---------------+-------------+-----------+
only showing top 5 rows


In [51]:
#Derniere question deja on télécharge ce qu"il faut 

comparison_year = 2026
filename = f"yellow_tripdata_{comparison_year}-01.parquet"
url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{filename}"

with requests.get(url, stream=True, timeout=120) as response:
    response.raise_for_status()
    with open(filename, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

df_recent = spark.read.parquet(filename)
df_recent.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [52]:
#on calcule les moyennes 
def average_metrics(df, year):
    return (
        df
        .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
        .filter(
            (F.col("trip_date") >= F.lit(f"{year}-01-01").cast("date"))
            & (F.col("trip_date") < F.lit(f"{year}-02-01").cast("date"))
        )
        .agg(
            F.avg("passenger_count").alias("average_passengers"),
            F.avg("trip_distance").alias("average_distance"),
            F.avg("fare_amount").alias("average_fare"),
            F.avg("tip_amount").alias("average_tip")
        )
        .withColumn("year", F.lit(year))
        .select(
            "year",
            "average_passengers",
            "average_distance",
            "average_fare",
            "average_tip"
        )
    )

In [53]:
#les deux années ensembles 
comparison = average_metrics(df_trips, 2019).unionByName(
    average_metrics(df_recent, comparison_year)
)

comparison.orderBy("year").show(truncate=False)

+----+------------------+-----------------+-----------------+------------------+
|year|average_passengers|average_distance |average_fare     |average_tip       |
+----+------------------+-----------------+-----------------+------------------+
|2019|1.5670089553079738|2.830113865760418|12.52951516096942|1.8208502016665093|
|2026|1.2562708015400346|6.455646629878646|20.80423360524865|2.6081446338431826|
+----+------------------+-----------------+-----------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [54]:
#On rend les DataFrames accessibles en SQL 
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [55]:
# Nombre de passagers moyen 

spark.sql("""
    SELECT AVG(passenger_count) AS average_passenger_count
    FROM trips
""").show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



In [56]:
#trajeets avec le plus de passagers 
spark.sql("""
    SELECT trip_id, passenger_count, trip_distance
    FROM trips
    ORDER BY passenger_count DESC NULLS LAST
    LIMIT 5
""").show()

+-----------+---------------+-------------+
|    trip_id|passenger_count|trip_distance|
+-----------+---------------+-------------+
|60131554242|            9.0|          0.0|
|60134539934|            9.0|          0.0|
|60132426139|            9.0|          0.0|
|60130492100|            9.0|          0.0|
|60134076851|            9.0|          0.0|
+-----------+---------------+-------------+



In [57]:
#borough avec le plus de depart avec jointure 
spark.sql("""
    SELECT
        z.Borough AS pickup_borough,
        COUNT(*) AS number_of_trips
    FROM trips AS t
    LEFT JOIN zones AS z
        ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY number_of_trips DESC
""").show()

+--------------+---------------+
|pickup_borough|number_of_trips|
+--------------+---------------+
|     Manhattan|        6950965|
|        Queens|         471173|
|       Unknown|         159815|
|      Brooklyn|          91905|
|         Bronx|          18062|
|           N/A|           3890|
|           EWR|            446|
| Staten Island|            361|
+--------------+---------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing